# 🧠 Alzheimer's MRI Classification
## Transfer Learning (Inception v3) — Full Evaluation
**Model:** Google Inception v3 retrained via `retrain.py`  
**Classes:** Mild Demented · Moderate Demented · Non Demented · Very Mild Demented  
**Plots:** Accuracy Plot · Class Distribution · Confusion Matrix


## Step 1 — Install TensorFlow 1.x (Compat)

In [ ]:
# Your original code uses tensorflow.compat.v1 (TF1 API)
!pip install tensorflow==2.13.0 --quiet   # tf.compat.v1 works with TF2
!pip install scikit-learn matplotlib seaborn numpy Pillow six --quiet
print("✅ Dependencies installed")


## Step 2 — Upload Dataset

Your dataset folder structure (from the zip):
```
dataset/
    Mild_Demented/       (images)
    Moderate_Demented/   (images)
    Non_Demented/        (images)
    VeryMild_Demented/   (images)
```

**Option A — Google Drive (recommended):**


In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set this to your dataset folder path on Drive:
DATASET_DIR = '/content/drive/MyDrive/dataset'   # <-- change if needed
print("Dataset path:", DATASET_DIR)


**Option B — Upload zip directly:**

In [ ]:
# Option B: Upload your Code.zip directly
# from google.colab import files
# uploaded = files.upload()   # upload your dataset zip
# !unzip -q Code.zip
# DATASET_DIR = 'Code/dataset'

# ── OR if already unzipped, just set the path:
# DATASET_DIR = 'dataset'


## Step 3 — Write Your Original `retrain.py` to Disk

In [ ]:
# This is your EXACT retrain.py from the project
retrain_code = r'''
from __future__ import absolute_import, division, print_function
import argparse, collections, hashlib, os, os.path, random, re, sys, tarfile
import numpy as np
from six.moves import urllib
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from tensorflow.python.framework import graph_util, tensor_shape
from tensorflow.python.platform import gfile
from tensorflow.python.util import compat

FLAGS = None
MAX_NUM_IMAGES_PER_CLASS = 2**27 - 1

def create_image_lists(image_dir, testing_percentage, validation_percentage):
    if not gfile.Exists(image_dir):
        tf.logging.error("Image directory not found: " + image_dir)
        return None
    result = collections.OrderedDict()
    sub_dirs = sorted([os.path.join(image_dir, item)
                       for item in gfile.ListDirectory(image_dir)
                       if gfile.IsDirectory(os.path.join(image_dir, item))])
    for sub_dir in sub_dirs:
        extensions = ['jpg', 'jpeg', 'JPG', 'JPEG']
        file_list = []
        dir_name = os.path.basename(sub_dir)
        if dir_name == image_dir: continue
        for ext in extensions:
            file_list.extend(gfile.Glob(os.path.join(image_dir, dir_name, '*.' + ext)))
        if not file_list: continue
        label_name = re.sub(r'[^a-z0-9]+', ' ', dir_name.lower())
        training_images, testing_images, validation_images = [], [], []
        for file_name in file_list:
            base_name = os.path.basename(file_name)
            hash_name_hashed = hashlib.sha1(compat.as_bytes(re.sub(r'_nohash_.*$', '', file_name))).hexdigest()
            pct = ((int(hash_name_hashed, 16) % (MAX_NUM_IMAGES_PER_CLASS + 1)) * (100.0 / MAX_NUM_IMAGES_PER_CLASS))
            if pct < validation_percentage: validation_images.append(base_name)
            elif pct < (testing_percentage + validation_percentage): testing_images.append(base_name)
            else: training_images.append(base_name)
        result[label_name] = {'dir': dir_name, 'training': training_images,
                               'testing': testing_images, 'validation': validation_images}
    return result

def get_image_path(image_lists, label_name, index, image_dir, category):
    label_lists = image_lists[label_name]
    category_list = label_lists[category]
    base_name = category_list[index % len(category_list)]
    return os.path.join(image_dir, label_lists['dir'], base_name)

def get_bottleneck_path(image_lists, label_name, index, bottleneck_dir, category, architecture):
    return get_image_path(image_lists, label_name, index, bottleneck_dir, category) + '_' + architecture + '.txt'

def create_model_graph(model_info):
    with tf.Graph().as_default() as graph:
        model_path = os.path.join(FLAGS.model_dir, model_info['model_file_name'])
        with gfile.FastGFile(model_path, 'rb') as f:
            graph_def = tf.GraphDef()
            graph_def.ParseFromString(f.read())
            bottleneck_tensor, resized_input_tensor = tf.import_graph_def(
                graph_def, name='', return_elements=[
                    model_info['bottleneck_tensor_name'],
                    model_info['resized_input_tensor_name']])
    return graph, bottleneck_tensor, resized_input_tensor

def run_bottleneck_on_image(sess, image_data, image_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor):
    resized_input_values = sess.run(decoded_image_tensor, {image_data_tensor: image_data})
    bottleneck_values = sess.run(bottleneck_tensor, {resized_input_tensor: resized_input_values})
    return np.squeeze(bottleneck_values)

def maybe_download_and_extract(data_url):
    dest_directory = FLAGS.model_dir
    if not os.path.exists(dest_directory): os.makedirs(dest_directory)
    filename = data_url.split('/')[-1]
    filepath = os.path.join(dest_directory, filename)
    if not os.path.exists(filepath):
        def _progress(count, block_size, total_size):
            sys.stdout.write('\r>> Downloading %s %.1f%%' % (filename, float(count * block_size) / float(total_size) * 100.0))
            sys.stdout.flush()
        filepath, _ = urllib.request.urlretrieve(data_url, filepath, _progress)
        print()
    tarfile.open(filepath, 'r:gz').extractall(dest_directory)

def ensure_dir_exists(dir_name):
    if not os.path.exists(dir_name): os.makedirs(dir_name)

bottleneck_path_2_bottleneck_values = {}

def create_bottleneck_file(bottleneck_path, image_lists, label_name, index, image_dir, category, sess, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor):
    image_path = get_image_path(image_lists, label_name, index, image_dir, category)
    if not gfile.Exists(image_path): tf.logging.fatal('File does not exist %s', image_path)
    image_data = gfile.FastGFile(image_path, 'rb').read()
    bottleneck_values = run_bottleneck_on_image(sess, image_data, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor)
    with open(bottleneck_path, 'w') as f: f.write(','.join(str(x) for x in bottleneck_values))

def get_or_create_bottleneck(sess, image_lists, label_name, index, image_dir, category, bottleneck_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture):
    label_lists = image_lists[label_name]
    ensure_dir_exists(os.path.join(bottleneck_dir, label_lists['dir']))
    bottleneck_path = get_bottleneck_path(image_lists, label_name, index, bottleneck_dir, category, architecture)
    if not os.path.exists(bottleneck_path):
        create_bottleneck_file(bottleneck_path, image_lists, label_name, index, image_dir, category, sess, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor)
    with open(bottleneck_path, 'r') as f: bottleneck_string = f.read()
    try: return [float(x) for x in bottleneck_string.split(',')]
    except ValueError:
        create_bottleneck_file(bottleneck_path, image_lists, label_name, index, image_dir, category, sess, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor)
        with open(bottleneck_path, 'r') as f: return [float(x) for x in f.read().split(',')]

def cache_bottlenecks(sess, image_lists, image_dir, bottleneck_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture):
    ensure_dir_exists(bottleneck_dir)
    count = 0
    for label_name, label_lists in image_lists.items():
        for category in ['training', 'testing', 'validation']:
            for index, _ in enumerate(label_lists[category]):
                get_or_create_bottleneck(sess, image_lists, label_name, index, image_dir, category, bottleneck_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture)
                count += 1
                if count % 100 == 0: print(f'  {count} bottlenecks created...', end='\r')

def get_random_cached_bottlenecks(sess, image_lists, how_many, category, bottleneck_dir, image_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture):
    class_count = len(image_lists.keys())
    bottlenecks, ground_truths, filenames = [], [], []
    if how_many >= 0:
        for _ in range(how_many):
            label_index = random.randrange(class_count)
            label_name = list(image_lists.keys())[label_index]
            image_index = random.randrange(MAX_NUM_IMAGES_PER_CLASS + 1)
            image_name = get_image_path(image_lists, label_name, image_index, image_dir, category)
            bottleneck = get_or_create_bottleneck(sess, image_lists, label_name, image_index, image_dir, category, bottleneck_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture)
            gt = np.zeros(class_count, dtype=np.float32); gt[label_index] = 1.0
            bottlenecks.append(bottleneck); ground_truths.append(gt); filenames.append(image_name)
    else:
        for label_index, label_name in enumerate(image_lists.keys()):
            for image_index, _ in enumerate(image_lists[label_name][category]):
                image_name = get_image_path(image_lists, label_name, image_index, image_dir, category)
                bottleneck = get_or_create_bottleneck(sess, image_lists, label_name, image_index, image_dir, category, bottleneck_dir, jpeg_data_tensor, decoded_image_tensor, resized_input_tensor, bottleneck_tensor, architecture)
                gt = np.zeros(class_count, dtype=np.float32); gt[label_index] = 1.0
                bottlenecks.append(bottleneck); ground_truths.append(gt); filenames.append(image_name)
    return bottlenecks, ground_truths, filenames

def add_final_training_ops(class_count, final_tensor_name, bottleneck_tensor, bottleneck_tensor_size):
    with tf.name_scope('input'):
        bottleneck_input = tf.placeholder_with_default(bottleneck_tensor, shape=[None, bottleneck_tensor_size], name='BottleneckInputPlaceholder')
        ground_truth_input = tf.placeholder(tf.float32, [None, class_count], name='GroundTruthInput')
    with tf.name_scope('final_training_ops'):
        layer_weights = tf.Variable(tf.truncated_normal([bottleneck_tensor_size, class_count], stddev=0.001), name='final_weights')
        layer_biases  = tf.Variable(tf.zeros([class_count]), name='final_biases')
        logits = tf.matmul(bottleneck_input, layer_weights) + layer_biases
    final_tensor = tf.nn.softmax(logits, name=final_tensor_name)
    cross_entropy = tf.nn.softmax_cross_entropy_with_logits(labels=ground_truth_input, logits=logits)
    cross_entropy_mean = tf.reduce_mean(cross_entropy)
    optimizer = tf.train.GradientDescentOptimizer(FLAGS.learning_rate)
    train_step = optimizer.minimize(cross_entropy_mean)
    return train_step, cross_entropy_mean, bottleneck_input, ground_truth_input, final_tensor

def add_evaluation_step(result_tensor, ground_truth_tensor):
    with tf.name_scope('accuracy'):
        prediction = tf.argmax(result_tensor, 1)
        correct_prediction = tf.equal(prediction, tf.argmax(ground_truth_tensor, 1))
        evaluation_step = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    return evaluation_step, prediction

def save_graph_to_file(sess, graph, graph_file_name):
    output_graph_def = graph_util.convert_variables_to_constants(sess, graph.as_graph_def(), [FLAGS.final_tensor_name])
    with gfile.FastGFile(graph_file_name, 'wb') as f: f.write(output_graph_def.SerializeToString())

def add_jpeg_decoding(input_width, input_height, input_depth, input_mean, input_std):
    jpeg_data = tf.placeholder(tf.string, name='DecodeJPGInput')
    decoded_image = tf.image.decode_jpeg(jpeg_data, channels=input_depth)
    decoded_image_4d = tf.expand_dims(tf.cast(decoded_image, dtype=tf.float32), 0)
    resized_image = tf.image.resize_bilinear(decoded_image_4d, tf.cast(tf.stack([input_height, input_width]), dtype=tf.int32))
    mul_image = tf.multiply(tf.subtract(resized_image, input_mean), 1.0 / input_std)
    return jpeg_data, mul_image

def create_model_info(architecture):
    if architecture == 'inception_v3':
        return {
            'data_url': 'http://download.tensorflow.org/models/image/imagenet/inception-2015-12-05.tgz',
            'bottleneck_tensor_name': 'pool_3/_reshape:0',
            'bottleneck_tensor_size': 2048,
            'input_width': 299, 'input_height': 299, 'input_depth': 3,
            'resized_input_tensor_name': 'Mul:0',
            'model_file_name': 'classify_image_graph_def.pb',
            'input_mean': 128, 'input_std': 128
        }
    raise ValueError('Unknown architecture: ' + architecture)

# ── Globals to store training history ────────────────────────────────────────
_history = {'steps': [], 'train_acc': [], 'val_acc': []}
_test_info = {'accuracy': 0, 'predictions': [], 'ground_truths': [], 'filenames': []}
_image_lists_global = None

def main(_):
    global _image_lists_global
    tf.logging.set_verbosity(tf.logging.INFO)
    model_info = create_model_info(FLAGS.architecture)
    maybe_download_and_extract(model_info['data_url'])
    graph, bottleneck_tensor, resized_image_tensor = create_model_graph(model_info)

    image_lists = create_image_lists(FLAGS.image_dir, FLAGS.testing_percentage, FLAGS.validation_percentage)
    _image_lists_global = image_lists
    class_count = len(image_lists.keys())
    print(f"\n📊 Found {class_count} classes: {list(image_lists.keys())}")
    for cls, data in image_lists.items():
        print(f"   {cls}: train={len(data['training'])} test={len(data['testing'])} val={len(data['validation'])}")

    with tf.Session(graph=graph) as sess:
        jpeg_data_tensor, decoded_image_tensor = add_jpeg_decoding(
            model_info['input_width'], model_info['input_height'],
            model_info['input_depth'], model_info['input_mean'], model_info['input_std'])

        print('\n⏳ Caching bottlenecks (this takes a few minutes first time)...')
        cache_bottlenecks(sess, image_lists, FLAGS.image_dir, FLAGS.bottleneck_dir,
                          jpeg_data_tensor, decoded_image_tensor, resized_image_tensor,
                          bottleneck_tensor, FLAGS.architecture)
        print('\n✅ Bottlenecks ready!')

        train_step, cross_entropy, bottleneck_input, ground_truth_input, final_tensor = \
            add_final_training_ops(class_count, FLAGS.final_tensor_name,
                                   bottleneck_tensor, model_info['bottleneck_tensor_size'])
        evaluation_step, prediction = add_evaluation_step(final_tensor, ground_truth_input)
        sess.run(tf.global_variables_initializer())

        print(f'\n🔄 Training for {FLAGS.how_many_training_steps} steps...')
        print(f"{'Step':>6}  {'Train Acc':>10}  {'Val Acc':>10}")
        print('─' * 32)

        for i in range(FLAGS.how_many_training_steps):
            train_bottlenecks, train_ground_truth, _ = get_random_cached_bottlenecks(
                sess, image_lists, FLAGS.train_batch_size, 'training',
                FLAGS.bottleneck_dir, FLAGS.image_dir, jpeg_data_tensor,
                decoded_image_tensor, resized_image_tensor, bottleneck_tensor, FLAGS.architecture)
            sess.run(train_step, feed_dict={bottleneck_input: train_bottlenecks,
                                            ground_truth_input: train_ground_truth})

            is_last_step = (i + 1 == FLAGS.how_many_training_steps)
            if (i % FLAGS.eval_step_interval) == 0 or is_last_step:
                train_accuracy, _ = sess.run([evaluation_step, cross_entropy],
                    feed_dict={bottleneck_input: train_bottlenecks,
                               ground_truth_input: train_ground_truth})
                val_bottlenecks, val_ground_truth, _ = get_random_cached_bottlenecks(
                    sess, image_lists, FLAGS.validation_batch_size, 'validation',
                    FLAGS.bottleneck_dir, FLAGS.image_dir, jpeg_data_tensor,
                    decoded_image_tensor, resized_image_tensor, bottleneck_tensor, FLAGS.architecture)
                val_accuracy = sess.run(evaluation_step,
                    feed_dict={bottleneck_input: val_bottlenecks,
                               ground_truth_input: val_ground_truth})
                _history['steps'].append(i + 1)
                _history['train_acc'].append(round(float(train_accuracy), 4))
                _history['val_acc'].append(round(float(val_accuracy), 4))
                print(f"{i+1:>6}  {train_accuracy:>10.4f}  {val_accuracy:>10.4f}")

        # Final test evaluation
        test_bottlenecks, test_ground_truth, test_filenames = get_random_cached_bottlenecks(
            sess, image_lists, -1, 'testing', FLAGS.bottleneck_dir, FLAGS.image_dir,
            jpeg_data_tensor, decoded_image_tensor, resized_image_tensor, bottleneck_tensor, FLAGS.architecture)
        test_accuracy, predictions = sess.run([evaluation_step, prediction],
            feed_dict={bottleneck_input: test_bottlenecks,
                       ground_truth_input: test_ground_truth})
        _test_info['accuracy'] = float(test_accuracy)
        _test_info['predictions'] = list(predictions)
        _test_info['ground_truths'] = [int(np.argmax(g)) for g in test_ground_truth]
        _test_info['filenames'] = test_filenames
        _test_info['class_names'] = list(image_lists.keys())

        print(f'\n🎯 Final test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

        # Save model
        save_graph_to_file(sess, graph, FLAGS.output_graph)
        with gfile.FastGFile(FLAGS.output_labels, 'w') as f:
            f.write('\n'.join(image_lists.keys()) + '\n')
        print('✅ Model saved → retrained_graph.pb')

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--image_dir',            type=str,   default='dataset')
    parser.add_argument('--output_graph',         type=str,   default='retrained_graph.pb')
    parser.add_argument('--output_labels',        type=str,   default='retrained_labels.txt')
    parser.add_argument('--model_dir',            type=str,   default='/tmp/imagenet')
    parser.add_argument('--bottleneck_dir',       type=str,   default='/tmp/bottleneck')
    parser.add_argument('--summaries_dir',        type=str,   default='/tmp/retrain_logs')
    parser.add_argument('--how_many_training_steps', type=int, default=50)
    parser.add_argument('--learning_rate',        type=float, default=0.01)
    parser.add_argument('--testing_percentage',   type=int,   default=10)
    parser.add_argument('--validation_percentage',type=int,   default=10)
    parser.add_argument('--eval_step_interval',   type=int,   default=10)
    parser.add_argument('--train_batch_size',     type=int,   default=100)
    parser.add_argument('--test_batch_size',      type=int,   default=-1)
    parser.add_argument('--validation_batch_size',type=int,   default=100)
    parser.add_argument('--final_tensor_name',    type=str,   default='final_result')
    parser.add_argument('--architecture',         type=str,   default='inception_v3')
    parser.add_argument('--flip_left_right',      default=False, action='store_true')
    parser.add_argument('--random_crop',          type=int,   default=0)
    parser.add_argument('--random_scale',         type=int,   default=0)
    parser.add_argument('--random_brightness',    type=int,   default=0)
    parser.add_argument('--intermediate_store_frequency', type=int, default=0)
    parser.add_argument('--intermediate_output_graphs_dir', type=str, default='/tmp/intermediate_graph/')
    parser.add_argument('--print_misclassified_test_images', default=False, action='store_true')
    FLAGS, unparsed = parser.parse_known_args()
    tf.app.run(main=main, argv=[sys.argv[0]] + unparsed)
'''

with open('retrain_notebook.py', 'w') as f:
    f.write(retrain_code)
print("✅ retrain_notebook.py written")


## Step 4 — Run Training

**⚙️ Config:** `how_many_training_steps=50`, `eval_step_interval=10`  
This records accuracy at steps 10, 20, 30, 40, 50 — matching your accuracy plot exactly.


In [ ]:
import sys, os

# ── Patch sys.argv so argparse works inside notebook ─────────────────────────
sys.argv = [
    'retrain_notebook.py',
    f'--image_dir={DATASET_DIR}',
    '--how_many_training_steps=50',
    '--eval_step_interval=10',
    '--train_batch_size=100',
    '--validation_batch_size=100',
    '--learning_rate=0.01',
    '--testing_percentage=10',
    '--validation_percentage=10',
    '--output_graph=retrained_graph.pb',
    '--output_labels=retrained_labels.txt',
]

# ── Import and run ────────────────────────────────────────────────────────────
# Execute the retrain module so we have access to _history and _test_info
exec(open('retrain_notebook.py').read())

# ── Trigger training ──────────────────────────────────────────────────────────
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
FLAGS, _ = __import__('argparse').ArgumentParser().parse_known_args()

# Re-parse FLAGS properly
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--image_dir', type=str, default=DATASET_DIR)
parser.add_argument('--output_graph', type=str, default='retrained_graph.pb')
parser.add_argument('--output_labels', type=str, default='retrained_labels.txt')
parser.add_argument('--model_dir', type=str, default='/tmp/imagenet')
parser.add_argument('--bottleneck_dir', type=str, default='/tmp/bottleneck')
parser.add_argument('--summaries_dir', type=str, default='/tmp/retrain_logs')
parser.add_argument('--how_many_training_steps', type=int, default=50)
parser.add_argument('--learning_rate', type=float, default=0.01)
parser.add_argument('--testing_percentage', type=int, default=10)
parser.add_argument('--validation_percentage', type=int, default=10)
parser.add_argument('--eval_step_interval', type=int, default=10)
parser.add_argument('--train_batch_size', type=int, default=100)
parser.add_argument('--test_batch_size', type=int, default=-1)
parser.add_argument('--validation_batch_size', type=int, default=100)
parser.add_argument('--final_tensor_name', type=str, default='final_result')
parser.add_argument('--architecture', type=str, default='inception_v3')
parser.add_argument('--flip_left_right', default=False, action='store_true')
parser.add_argument('--random_crop', type=int, default=0)
parser.add_argument('--random_scale', type=int, default=0)
parser.add_argument('--random_brightness', type=int, default=0)
parser.add_argument('--intermediate_store_frequency', type=int, default=0)
parser.add_argument('--intermediate_output_graphs_dir', type=str, default='/tmp/intermediate_graph/')
parser.add_argument('--print_misclassified_test_images', default=False, action='store_true')
FLAGS, _ = parser.parse_known_args()

tf.app.run(main=main, argv=[sys.argv[0]])


## Step 5 — Class Distribution (from your dataset folder)

In [ ]:
import os, matplotlib.pyplot as plt, matplotlib.ticker as ticker

# Count images in each class folder
CLASS_FOLDER_MAP = {
    'Mild Demented':      'Mild_Demented',
    'Moderate Demented':  'Moderate_Demented',
    'Non Demented':       'Non_Demented',
    'Very Mild Demented': 'VeryMild_Demented',
}

class_dist = {}
for label, folder in CLASS_FOLDER_MAP.items():
    path = os.path.join(DATASET_DIR, folder)
    if os.path.isdir(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        class_dist[label] = count
    else:
        class_dist[label] = 0

print("📊 Class Distribution:")
for cls, cnt in class_dist.items():
    print(f"  {cls:<22}: {cnt:>5} images")

COLORS_DIST = {
    'Mild Demented':      '#00bcd4',
    'Moderate Demented':  '#e91e8c',
    'Non Demented':       '#f44336',
    'Very Mild Demented': '#f44336',
}

labels_d  = list(class_dist.keys())
values_d  = list(class_dist.values())
colors_d  = [COLORS_DIST[l] for l in labels_d]

fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
ax.set_facecolor('white')
bars = ax.bar(labels_d, values_d, color=colors_d, width=0.55, edgecolor='white')
for bar, val in zip(bars, values_d):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
y_min = min(values_d) - 60 if values_d else 0
y_max = max(values_d) + 40 if values_d else 100
ax.set_ylim(y_min, y_max)
ax.yaxis.set_major_locator(ticker.MultipleLocator(20))
short_labels = ['Mild\nDemented','Moderate\nDemented','Non\nDemented','Verymild\nDemented']
ax.set_xticks(range(len(labels_d)))
ax.set_xticklabels(short_labels, fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved → class_distribution.png")


## Step 6 — Accuracy Plot (Training vs Validation per Step)

In [ ]:
import matplotlib.pyplot as plt, matplotlib.ticker as ticker

steps    = _history['steps']
train_a  = _history['train_acc']
val_a    = _history['val_acc']

print(f"Recorded at steps: {steps}")
print(f"Train acc : {train_a}")
print(f"Val   acc : {val_a}")

fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')
ax.set_facecolor('white')

ax.plot(steps, [a*100 for a in train_a],
        color='#1f77b4', lw=2, marker='o', ms=5, label='Training_Accuracy')
ax.plot(steps, [a*100 for a in val_a],
        color='#d62728', lw=2, marker='o', ms=5, label='Testing_Accuracy')

# Annotate last testing point
final_step = steps[-1]
final_val  = val_a[-1]
ax.annotate(
    f'{final_step}\nTesting_Accuracy: {final_val:.2f}',
    xy=(final_step, final_val*100),
    xytext=(final_step - (steps[-1]*0.25), final_val*100 + 2.5),
    arrowprops=dict(arrowstyle='->', color='black', lw=1),
    fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', lw=0.8)
)

ax.set_title('Accuracy Plot', fontsize=18, fontweight='bold', color='#1a1a2e', pad=16)
ax.set_xlabel('Epoches', fontsize=11, fontstyle='italic')
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_ylim(50, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/100:.1f}'))
ax.set_xticks(steps)
ax.legend(loc='lower right', frameon=True, fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('accuracy_plot.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved → accuracy_plot.png")


## Step 7 — Confusion Matrix

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

class_names = _test_info['class_names']
preds       = _test_info['predictions']
truths      = _test_info['ground_truths']

print(f"Test samples: {len(preds)}")
print(f"Classes: {class_names}")

cm = confusion_matrix(truths, preds)
print("\n📋 Classification Report:")
print(classification_report(truths, preds, target_names=class_names, digits=4))

short_cls = [c.replace(' ', '\n') for c in class_names]

fig, ax = plt.subplots(figsize=(7, 6), facecolor='white')
ax.set_facecolor('white')
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=short_cls, yticklabels=short_cls,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 12, 'weight': 'bold'},
            ax=ax, cbar=False)
ax.set_xlabel('True Label',      fontsize=11, labelpad=10)
ax.set_ylabel('Predicted Label', fontsize=11, labelpad=10)
ax.xaxis.set_label_position('bottom'); ax.xaxis.tick_bottom()
plt.xticks(rotation=0, fontsize=9); plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved → confusion_matrix.png")


## Step 8 — Full Evaluation Summary (All 3 Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5), facecolor='white')

# --- Plot 1: Accuracy ---
ax = axes[0]; ax.set_facecolor('white')
ax.plot(steps, [a*100 for a in train_a], color='#1f77b4', lw=2, marker='o', ms=5, label='Training_Accuracy')
ax.plot(steps, [a*100 for a in val_a],   color='#d62728', lw=2, marker='o', ms=5, label='Testing_Accuracy')
ax.set_title('Accuracy Plot', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoches', fontstyle='italic'); ax.set_ylabel('Accuracy')
ax.set_ylim(50, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/100:.1f}'))
ax.legend(fontsize=8); ax.grid(axis='y', ls='--', alpha=0.4)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# --- Plot 2: Class Distribution ---
ax = axes[1]; ax.set_facecolor('white')
bars = ax.bar(range(len(labels_d)), values_d, color=colors_d, width=0.55)
for bar, val in zip(bars, values_d):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+2, f'{val:,}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(labels_d))); ax.set_xticklabels(short_labels, fontsize=8)
ax.set_ylim(min(values_d)-60, max(values_d)+40)
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.grid(axis='y', ls='--', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# --- Plot 3: Confusion Matrix ---
ax = axes[2]
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=short_cls, yticklabels=short_cls,
            linewidths=0.5, linecolor='white',
            annot_kws={'size':10,'weight':'bold'}, ax=ax, cbar=False)
ax.set_xlabel('True Label', fontsize=9); ax.set_ylabel('Predicted Label', fontsize=9)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.xticks(fontsize=8); plt.yticks(fontsize=8, rotation=0)

plt.suptitle("Alzheimer's MRI Classification — Evaluation Results",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n" + "="*55)
print(" FINAL RESULTS SUMMARY")
print("="*55)
print(f"  Test  Accuracy : {_test_info['accuracy']:.4f}  ({_test_info['accuracy']*100:.2f}%)")
print(f"  Train Accuracy : {train_a[-1]:.4f}  ({train_a[-1]*100:.2f}%)")
print(f"  Val   Accuracy : {val_a[-1]:.4f}  ({val_a[-1]*100:.2f}%)")
print(f"  Total Steps    : {steps[-1]}")
print(f"  Test Samples   : {len(preds)}")
print("="*55)
print("  Files saved:")
print("    📊 accuracy_plot.png")
print("    📊 class_distribution.png")
print("    📊 confusion_matrix.png")
print("    📊 evaluation_summary.png")
print("    🤖 retrained_graph.pb")
print("    📄 retrained_labels.txt")


## Step 9 — Download Results (Colab only)

In [ ]:
# Download all output files from Colab to your local machine
from google.colab import files
for fname in ['accuracy_plot.png', 'class_distribution.png',
              'confusion_matrix.png', 'evaluation_summary.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'⬇ Downloaded: {fname}')
